# Introduction to geocoding: addresses ↔ coordinates

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/oechenique/Geocoding/blob/main/Notebook/01_Intro_To_Geocoding.ipynb)

Forward geocoding (address → point) and reverse geocoding (point → address) with [Nominatim](https://nominatim.org/), the OpenStreetMap geocoder, through `geopy`.

What this notebook does, on purpose with very few requests:
1. Geocodes three Madrid museums, first as free text and then as a **structured query**.
2. Reverse-geocodes six points and measures **how complete** the returned addresses are (street, house number, postcode).
3. Puts everything on an interactive map.

Nominatim's [usage policy](https://operations.osmfoundation.org/policies/nominatim/) asks for **at most 1 request per second**, a **User-Agent that identifies the app** and caching. No bulk geocoding on the public server.

In [ ]:
import sys

if "google.colab" in sys.modules:  # explore() needs mapclassify, which Colab may not have
    %pip install -q mapclassify==2.11.0

In [1]:
# Everything needed is already on Colab (geopy, geopandas, folium). Locally: requirements.txt.
import pandas as pd
import geopandas as gpd
from geopy.geocoders import Nominatim
from geopy.extra.rate_limiter import RateLimiter

## 1. A polite geocoder

A real User-Agent (not `my_app` or `your_project_name`, which Nominatim may block) and a `RateLimiter` that waits 1 s between calls and retries twice on errors. Results go to a small JSON cache, so a second run of the notebook sends no requests.

In [2]:
USER_AGENT = "oechenique-geocoding-tutorial/1.0 (+https://github.com/oechenique/Geocoding)"

nominatim = Nominatim(user_agent=USER_AGENT, timeout=10)
geocode = RateLimiter(nominatim.geocode, min_delay_seconds=1, max_retries=2)
reverse = RateLimiter(nominatim.reverse, min_delay_seconds=1, max_retries=2)

In [3]:
import json
from pathlib import Path

CACHE = Path("cache/geocoding.json")  # out of git: re-running the notebook does not hit the APIs again


def cached(key, fn):
    """Return fn() from a JSON cache on disk. Only JSON-serializable results go in."""
    store = json.loads(CACHE.read_text(encoding="utf-8")) if CACHE.exists() else {}
    if key not in store:
        store[key] = fn()
        CACHE.parent.mkdir(exist_ok=True)
        CACHE.write_text(json.dumps(store, ensure_ascii=False, indent=1), encoding="utf-8")
    return store[key]

## 2. Forward geocoding: free text vs structured query

The same three museums (the example from the 2023 version of this repo). A free-text query lets Nominatim guess which part is the street and which the city; a **structured query** says it explicitly, which usually helps with house numbers.

In [4]:
museums = pd.DataFrame({
    "museum": ["Museo del Prado", "Museo Reina Sofía", "Museo Thyssen-Bornemisza"],
    "street": ["Paseo del Prado", "52 Calle de Santa Isabel", "8 Paseo del Prado"],
    "postalcode": ["28014", "28012", "28014"],
})
museums["address"] = museums["street"] + ", " + museums["postalcode"] + " Madrid, Spain"


def forward(query):
    loc = geocode(query, addressdetails=True)
    return None if loc is None else {"lat": loc.latitude, "lon": loc.longitude,
                                     "display_name": loc.address, "type": loc.raw.get("type")}


free = [cached(f"free|{a}", lambda a=a: forward(a)) for a in museums["address"]]
structured = [
    cached(f"struct|{r.street}|{r.postalcode}",
           lambda r=r: forward({"street": r.street, "postalcode": r.postalcode, "city": "Madrid", "country": "Spain"}))
    for r in museums.itertuples()
]

for name, a, b in zip(museums["museum"], free, structured):
    print(f"{name}\n  free text : {a and a['display_name']}\n  structured: {b and b['display_name']}")

Museo del Prado
  free text : Paseo del Prado, Cortes, Centro, Madrid, Comunidad de Madrid, 28014, España
  structured: Paseo del Prado, Cortes, Centro, Madrid, Comunidad de Madrid, 28014, España
Museo Reina Sofía
  free text : Museo Nacional Centro de Arte Reina Sofía, 52, Calle de Santa Isabel, Embajadores, Centro, Madrid, Comunidad de Madrid, 28012, España
  structured: Museo Nacional Centro de Arte Reina Sofía, 52, Calle de Santa Isabel, Embajadores, Centro, Madrid, Comunidad de Madrid, 28012, España
Museo Thyssen-Bornemisza
  free text : Museo Thyssen-Bornemisza, 8, Paseo del Prado, Cortes, Centro, Madrid, Comunidad de Madrid, 28014, España
  structured: Museo Thyssen-Bornemisza, 8, Paseo del Prado, Cortes, Centro, Madrid, Comunidad de Madrid, 28014, España


In [5]:
museums["lat"] = [r["lat"] if r else None for r in structured]
museums["lon"] = [r["lon"] if r else None for r in structured]
museums_gdf = gpd.GeoDataFrame(museums, geometry=gpd.points_from_xy(museums["lon"], museums["lat"]), crs="EPSG:4326")

# How far apart are the two answers? Distances in metres need a projected CRS (UTM 30N for Madrid).
free_gdf = gpd.GeoSeries(gpd.points_from_xy([r["lon"] for r in free], [r["lat"] for r in free]), crs="EPSG:4326")
museums["free_vs_structured_m"] = free_gdf.to_crs(25830).distance(museums_gdf.to_crs(25830).geometry).round(0)
museums[["museum", "lat", "lon", "free_vs_structured_m"]]

,museum,lat,lon,free_vs_structured_m
0,Museo del Prado,40.415300,-3.694544,672.0
1,Museo Reina Sofía,40.408049,-3.694422,0.0
2,Museo Thyssen-Bornemisza,40.416216,-3.694932,0.0


## 3. Reverse geocoding and address completeness

Six points around central Madrid. `addressdetails` returns the address broken into fields; the question is how many of them come back filled. A point in a park has no house number, and that is correct, not an error.

In [6]:
points = pd.DataFrame({
    "label": ["Puerta del Sol", "Retiro park (pond)", "Gran Vía 28", "Lavapiés square", "Chamberí residential block", "Atocha station"],
    "lat": [40.41694, 40.41794, 40.42013, 40.40883, 40.43448, 40.40652],
    "lon": [-3.70347, -3.68226, -3.70287, -3.70081, -3.70451, -3.68943],
})


def backward(lat, lon):
    loc = reverse((lat, lon), addressdetails=True, zoom=18)
    return None if loc is None else {"display_name": loc.address, **loc.raw.get("address", {})}


rev = [cached(f"rev|{r.lat}|{r.lon}", lambda r=r: backward(r.lat, r.lon)) for r in points.itertuples()]
fields = ["road", "house_number", "postcode", "suburb", "city"]
addr = pd.DataFrame([{f: (r or {}).get(f) for f in fields} for r in rev])
points = pd.concat([points, addr], axis=1)
points[["label"] + fields]

,label,road,house_number,postcode,suburb,city
0,Puerta del Sol,Plaza de la Puerta del Sol,11,28013,NaN,Madrid
1,Retiro park (pond),Paseo de la República Dominicana,NaN,28001,Retiro,Madrid
2,Gran Vía 28,Gran Vía,30,28013,NaN,Madrid
3,Lavapiés square,Plaza de Lavapiés,5,28012,NaN,Madrid
4,Chamberí residential block,Calle de Bravo Murillo,7,28015,Chamberí,Madrid
5,Atocha station,Intercambiador de Atocha,NaN,28014,Arganzuela,Madrid


In [7]:
completeness = (addr.notna().mean() * 100).round(0).rename("% filled").to_frame()
completeness

,% filled
road,100.0
house_number,67.0
postcode,100.0
suburb,50.0
city,100.0


## 4. Map

`GeoDataFrame.explore()` builds a folium map: museums in red, reverse-geocoded points in blue.

In [8]:
points_gdf = gpd.GeoDataFrame(points, geometry=gpd.points_from_xy(points["lon"], points["lat"]), crs="EPSG:4326")
m = museums_gdf[["museum", "address", "geometry"]].explore(color="crimson", marker_kwds={"radius": 8}, tiles="OpenStreetMap")
points_gdf[["label", "road", "house_number", "postcode", "geometry"]].explore(m=m, color="royalblue", marker_kwds={"radius": 6})

## Takeaways

- With a house number (Reina Sofía, Thyssen) free text and structured query land on the **same building, 0 m apart**. Without one ("Paseo del Prado"), both return **the street, not the museum**, and pick different segments of it: **672 m apart**. A geocoder answers the question you asked, not the one you meant.
- 6 points reverse-geocoded: road, postcode and city came back for all 6, house number for 4 (the park and the station have none, which is correct) and suburb for 3.
- Reverse geocoding rarely fills every field; check completeness before joining on postcode or house number.
- Next: [02_Geocoding_With_Apis.ipynb](02_Geocoding_With_Apis.ipynb) compares two OSM geocoders (Nominatim and Photon) against reference coordinates.

Data © [OpenStreetMap contributors](https://www.openstreetmap.org/copyright), ODbL.